# Styling

One vocabulary on every `add_*` method — `color`, `fill_color`, `fill_opacity`,
`weight`, `opacity`, `radius` where it applies (Leaflet's camelCase spellings work
too). This notebook goes from layer-wide options to per-feature styling from the
data, then to driving color and size through a column.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(11)
n = 120
df = pd.DataFrame({
    "lat": 36.05 + rng.normal(0, 0.05, n),
    "lon": -5.45 + rng.normal(0, 0.08, n),
    "site": [f"S{i:03d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "volume": rng.integers(10, 500, n),
    "kind": rng.choice(["city", "town", "village"], n),
})
ring = [[36.00, -5.62], [36.00, -5.50], [36.08, -5.50], [36.08, -5.62]]

## Layer options

Polygons follow Leaflet semantics: the fill reads `fill_color` (defaulting to
`color` when unset), the border draws from `color` + `weight` + `opacity`.

In [ ]:
m = Map()
m.add_polygon(ring, name="Zone", color="crimson",
              fill_color="#f5c4ac", fill_opacity=0.45, weight=4)
m

Point layers take their **alpha from the color itself** — `#rrggbbaa` or
`rgba(...)` — rather than from `opacity`:

In [ ]:
m = Map()
m.add_circle_markers(df, name="Translucent", color="#e15759aa", radius=9)
m

## Per-feature styling from the data

A property or column named exactly `style` is applied per feature. The value is a
color string or a dict of options. (Only that exact name is claimed — a column
called `color` is data, not styling.)

In [ ]:
styled = df.copy()
styled["style"] = styled["kind"].map({
    "city": "#e15759",
    "town": {"color": "#4e79a7", "radius": 6},
    "village": {"color": "#59a14f", "radius": 4},
})
m = Map()
m.add_circle_markers(styled, name="By kind")
m

## Precedence

`static_style` → explicit keyword → `style` column → defaults. So `static_style`
forces one appearance regardless of the data, and a plain keyword still beats the
column:

In [ ]:
m = Map()
m.add_circle_markers(styled, name="Forced grey", static_style={"color": "grey"})
m.add_circle_markers(styled, name="Keyword wins", color="purple",
                     layer_group="Comparison")
m

## Honest warnings

A misspelled option is reported with a suggestion instead of silently rendering
the default. A real option the geometry cannot draw says so too — those are
different mistakes and the messages tell them apart.

In [ ]:
import warnings

m = Map()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    m.add_circle_markers(df, name="Typo", colour="red")      # one edit from 'color'
    m.add_circle_markers(df, name="Undrawn", weight=5)       # weight is lines/areas
[str(w.message) for w in caught]

## Color from a column

`color_col` colors features through a colormap — numeric columns ramp between the
data's extremes (or `vmin`/`vmax` when you fix them). Available ramps: `viridis`
(default), `plasma`, `inferno`, `magma`, `turbo`, `coolwarm`, `blues`, `reds`,
`greens`, `greys`.

In [ ]:
m = Map()
m.add_circle_markers(df, name="Reading", color_col="reading",
                     colormap="plasma", vmin=0, vmax=40)
m

`color_bins` classifies into discrete classes instead of a continuous ramp — the
usual choropleth move:

In [ ]:
m = Map()
m.add_circle_markers(df, name="Binned", color_col="reading",
                     color_bins=[10, 20, 30])
m

A **non-numeric** column is categorical: each distinct value takes a palette color
(`swift10`). Naming a sequential colormap instead spreads it evenly across the
categories:

In [ ]:
m = Map()
m.add_circle_markers(df, name="Categories", color_col="kind")
m.add_circle_markers(df, name="Categories, ramped", color_col="kind",
                     colormap="blues", layer_group="Comparison")
m

## Size from a column

`radius_col` sizes points so **area** is proportional to the value, across
`radius_range` pixels. Missing values take the smallest radius; missing values in
`color_col` paint as the layer's base color.

In [ ]:
m = Map()
m.add_circle_markers(df, name="Bubbles", color_col="reading",
                     radius_col="volume", radius_range=(3, 18))
m

Data-driven styling works on every geometry: lines take the ramp on their stroke,
polygons take it on their **fill** while the border keeps `color` — a choropleth.

Transient styling — highlighting a selection, restyling one feature — is
**05_layer_control**'s topic: it layers *above* everything set here and clears
without touching it.

## The legend picks all of this up

Every mapping in this notebook records its legend block at add time — ramps,
bins, categories, and a stated size row for `radius_col` (`size ∝ volume
(10 – 500)`; stated, never drawn — legend pixels are not map pixels at any
zoom). One call shows it on the bubble map above; **11_legend** is the full tour:

In [ ]:
m.configure_legend(show=True, title="Styling demo");